In [1]:
import re
import numpy as np
import random
import os
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
seed = 42

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

In [3]:
# ───데이터 ───────────────────────────────────────────────
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    return [line.strip() for line in lines if line.strip()]

In [4]:
# ───파일 로드 ───────────────────────────────────────────────
raw_texts = load_data("input_data.txt")
targets = load_data("target_data.txt")

print("raw_texts:", raw_texts[:3])
print("targets:", targets[:3])

raw_texts: ['Wed 11:00 appointment', 'Mon 21:00 reading', 'Fri 12:00 lunch']
targets: ['수요일 11시 약속', '월요일 21시 독서', '금요일 12시 점심']


In [5]:
# ───char-level 토크나이저 ─────────────────────────────────
start_token = '\t'
end_token   = '\n'

decoder_texts = [start_token + t + end_token for t in targets]

tokenizer = Tokenizer(char_level=True, filters='', lower=False)
tokenizer.fit_on_texts(raw_texts + decoder_texts)

In [6]:
#과제 : 토크나이징 및 패딩 진행
src_len = 20
tgt_len = 25

# encoder input (raw_texts)
input_seq = tokenizer.texts_to_sequences(raw_texts)
input_ids = pad_sequences(input_seq, maxlen=src_len, padding='post')

# decoder input (decoder_texts)
decoder_seq = tokenizer.texts_to_sequences(decoder_texts)
decoder_input_ids = pad_sequences(decoder_seq, maxlen=tgt_len, padding='post')

# decoder target (targets - 한글 데이터)
target_seq = tokenizer.texts_to_sequences(targets)
decoder_target_ids = pad_sequences(target_seq, maxlen=tgt_len, padding='post')

print("input_ids shape:", input_ids.shape)
print("decoder_input_ids shape:", decoder_input_ids.shape)
print("decoder_target_ids shape:", decoder_target_ids.shape)

# char to index, index to char 매핑
char_to_index = tokenizer.word_index
index_to_char = {v: k for k, v in char_to_index.items()}

start_id = char_to_index.get(start_token, 1)
end_id = char_to_index.get(end_token, 2)
vocab_size = len(char_to_index) + 1

print(f"start_id: {start_id}, end_id: {end_id}, vocab_size: {vocab_size}")

input_ids shape: (200, 20)
decoder_input_ids shape: (200, 25)
decoder_target_ids shape: (200, 25)
start_id: 8, end_id: 11, vocab_size: 65


In [7]:
#과제 : LSTM 기반 seq2seq 모델 구축하기
latent_dim = 128

# Encoder
encoder_inputs = Input(shape=(src_len,))
enc_emb = Embedding(vocab_size, 64)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(tgt_len,))
dec_emb_layer = Embedding(vocab_size, 64)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

print("Model architecture built successfully")

Model architecture built successfully


In [8]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 25)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 64)    │      4,160 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 25, 64)    │      4,160 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │     98,816 │ embedding[0][0]   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 25, 128), │     98,816 │ embedding_1[0][0… │
│                     │ (None, 128),      │            │ lstm[0][1],       │
│                     │ (None, 128)]      │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 25, 65)    │      8,385 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 214,337 (837.25 KB)

 Trainable params: 214,337 (837.25 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# ───학습 ─────────────────────────────────────────────────
decoder_target_ids_exp = np.expand_dims(decoder_target_ids, -1)   # (B, tgt_len, 1)

model.fit(
    [input_ids, decoder_input_ids],
    decoder_target_ids_exp,
    batch_size=4,
    epochs=50,
    verbose=1,
)
print("학습 완료")

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6642 - loss: 1.9305
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6642 - loss: 1.9305
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7870 - loss: 0.8815
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7870 - loss: 0.8815
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8318 - loss: 0.6457
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8318 - loss: 0.6457
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8372 - loss: 0.5092
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8372 - loss: 0.5092
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8458 - loss: 0.4474
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8458 - loss: 0.4474
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8564 - loss: 0.4138
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8564 - loss:

In [10]:
# ───추론 모델 (inference model) ──────────────────────────
# Encoder inference
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=decoder_states_inputs
)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model([decoder_inputs] + decoder_states_inputs,[decoder_outputs2] + decoder_states2)

In [11]:
# ───Decode ─────────────────────────────────────────
def greedy_decode(text, max_len=20):
    seq = tokenizer.texts_to_sequences([text])
    ids = pad_sequences(seq, maxlen=src_len, padding='post')

    h, c = encoder_model.predict(ids, verbose=0)
    token = np.array([[start_id]])
    out_chars = []

    for _ in range(max_len):
        prob, h, c = decoder_model.predict([token, h, c], verbose=0)
        next_id = int(np.argmax(prob[0, -1]))
        if next_id == end_id or next_id == 0:
            break
        out_chars.append(index_to_char.get(next_id, ''))
        token = np.array([[next_id]])

    return ''.join(out_chars)

In [12]:
# ───테스트 데이터────────────────────────────────────────────────
test_cases = [
"Mon 15:00 movie",
"Fri 08:00 appointment",
"Wed 13:00 study",
"Thu 18:00 game",
"Mon 13:00 lunch",
]

In [13]:
# ───요일 순서────────────────────────────────────────────────
day_order = {
    "월요일": 0,
    "화요일": 1,
    "수요일": 2,
    "목요일": 3,
    "금요일": 4,
    "토요일": 5,
    "일요일": 6
}
decoded_outputs = []

for t in test_cases:
    pred = greedy_decode(t)
    print(f"입력: {t}")
    print(f"출력: {pred}\n")

    decoded_outputs.append(pred)

입력: Mon 15:00 movie
출력: 월요일 13시 영화

입력: Fri 08:00 appointment
출력: 금요일 08시 약속

입력: Fri 08:00 appointment
출력: 금요일 08시 약속

입력: Wed 13:00 study
출력: 수요일 15시 공부

입력: Wed 13:00 study
출력: 수요일 15시 공부

입력: Thu 18:00 game
출력: 월요일 18시 게임

입력: Thu 18:00 game
출력: 월요일 18시 게임

입력: Mon 13:00 lunch
출력: 월요일 13시 점심

입력: Mon 13:00 lunch
출력: 월요일 13시 점심



In [14]:
# ───스케쥴 파싱 함수 작성────────────────────────────────────────────────
def parse_schedule(text):
    pattern = r"^(월요일|화요일|수요일|목요일|금요일|토요일|일요일)\s+(\d{1,2})시(?:(\d{1,2})분)?\s+(.+)$"
    match = re.match(pattern, text.strip())

    if not match:
        return None

    day = match.group(1)
    hour = int(match.group(2))
    minute = int(match.group(3)) if match.group(3) else 0
    event = match.group(4).strip()

    return (day, hour, minute, event, text)

In [15]:
#과제 : 스케쥴 정렬
parsed_schedules = []

for output in decoded_outputs:
    parsed = parse_schedule(output)
    if parsed:
        parsed_schedules.append(parsed)

# 요일 순서 -> 시간 순서로 정렬
parsed_schedules.sort(key=lambda x: (day_order[x[0]], x[1], x[2]))

print("정렬된 스케줄:")
for day, hour, minute, event, original_text in parsed_schedules:
    print(f"{day} {hour:02d}:{minute:02d} {event}")

정렬된 스케줄:
월요일 13:00 영화
월요일 13:00 점심
월요일 18:00 게임
수요일 15:00 공부
금요일 08:00 약속


In [16]:
with open("output.txt", "w", encoding="utf-8") as f:
    for day, hour, minute, event, original_text in parsed_schedules:
        f.write(original_text + "\n")